In [ ]:
from getpass import getuser # Libaray to copy things
from pathlib import Path # Object oriented libary to deal with paths
from dask.utils import format_bytes
from distributed import Client, progress, wait # Libaray to orchestrate distributed resources
#from dask_jobqueue import SLURMCluster # Setting up distributed memories via slurm
import numpy as np # Pythons standard array library
import xarray as xr # Libary to work with labeled n-dimensional data
import dask # Distributed data libary
import dask.distributed
import multiprocessing
from dask.diagnostics import ProgressBar
import glob, os
from matplotlib.lines import Line2D
from scipy import stats

from esda.moran import Moran
from libpysal.weights import KNN
from matplotlib.patches import Patch
from scipy.ndimage import gaussian_filter1d
from subprocess import run, PIPE
import sys
import os
from sklearn.neighbors import KernelDensity
import gc
from concurrent.futures import ProcessPoolExecutor
import seaborn as sns
import xskillscore as xs
import warnings
warnings.filterwarnings(action='ignore')
dask.config.config.get('distributed').get('dashboard').update({'link':'{JUPYTERHUB_SERVICE_PREFIX}/proxy/{port}/status'})
import pandas as pd
import re

from collections import defaultdict
from scipy.interpolate import CloughTocher2DInterpolator, LinearNDInterpolator, NearestNDInterpolator

In [ ]:
#'''Provides information about the CPU and starts the dask client'''
ncpu = multiprocessing.cpu_count()
processes = False
nworker = 4
threads = ncpu // nworker
print(f"Number of CPUs: {ncpu}, number of threads: {threads}, number of workers: {nworker}, processes: {processes}")
client = Client(processes=processes,
 threads_per_worker=threads,
 n_workers=nworker,
 memory_limit='1000GB'
 )

In [ ]:
client

In [ ]:
def temporal_auto(data,model):
    # 4. Create a lagged version (shift by 1 day)
    data_lag1 = data.shift(time=1)
    # 5. Compute lag-1 autocorrelation
    #    Definition: corr(binary(t), binary(t−1)) along time dimension
    r_pear = xr.corr(data, data_lag1, dim='time')

    return r_pear
    

## IFS deep on

In [ ]:
#IFS deep on land
latitude = [30,60]
model = "IFS"

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_on/daily_precip')
files = sorted([str(f) for f in data_path.glob('IFS_2.8-FESOM_5_daily_precip_*_2020.nc')])[:]
ifs_on = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']
indexer = ((ifs_on.lat>=latitude[0]) & (ifs_on.lat<=latitude[1])).compute()
ifs_on = ifs_on.where(indexer,drop=True)
print('hello')
print(ifs_on)
temporal_temp_pear = temporal_auto(ifs_on,model)
temporal_temp_pear = temporal_temp_pear.load()

temporal_temp_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_2.8km_on_30-60N_temporal_corr_nonbinary.nc')




In [ ]:
print(temporal_temp_spear.sum())

## IFS deep off

In [ ]:
latitude = [30,60]
model = "IFS"

data_path = Path('/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_2.8km_deep_off/daily_precip')
files = sorted([str(f) for f in data_path.glob('IFS_2.8-FESOM_5_deep_off_daily_precip_*_2020.nc')])[:]
ifs_off = xr.open_mfdataset(files, engine='netcdf4',combine='by_coords')['tp']
indexer = ((ifs_off.lat>=latitude[0]) & (ifs_off.lat<=latitude[1])).compute()
ifs_off = ifs_off.where(indexer,drop=True)
print('hello')
temporal_temp_pear = temporal_auto(ifs_off,model)
temporal_temp_pear = temporal_temp_pear.load()
temporal_temp_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_2.8km_off_30-60N_temporal_corr_nonbinary.nc')


## IFS cycle3

In [ ]:
model = "IFS"
years = range(2020, 2025)

results_temporal_pear = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_4.4-FESOM_5/daily_precip/daysum_IFS_4.4-FESOM_5_precip_30-60N_{year}.nc")    
    dry_da = ds["tp"]
    temporal_temp_pear = temporal_auto(dry_da,model)
    temporal_temp_pear = temporal_temp_pear.load()
    temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [year]})

    results_temporal_pear.append(temporal_temp_pear)

ifs_cycle3_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

ifs_cycle3_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_cycle3_30-60N_temporal_corr_nonbinary.nc')



## IFS cycle4

In [ ]:
model = "IFS"
#years = range(1990, 2020)
years = range(2020, 2050)
#years = range(2020, 2021)

months = [5,6,7,8,9,10]

results_temporal_pear = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/30-60N_IFS_9-FESOM_5_daily_precip_{year}.nc")
    #ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/IFS_zoom9/2020-2050/daily_precip/wet_30-60N_merged_{year}.nc")
    ds = ds.where(ds.time.dt.month.isin(months), drop=True)
    dry_da = ds["tp"]
    temporal_temp_pear = temporal_auto(dry_da,model)
    temporal_temp_pear = temporal_temp_pear.load()
    temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [year]})

    results_temporal_pear.append(temporal_temp_pear)

ifs_cycle4_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")

ifs_cycle4_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/ifs_cycle4_2020-2050_30-60N_temporal_corr_nonbinary.nc')





## ICON cycle3

In [ ]:
model = "ICON"
years = range(2020, 2025)

results_temporal_pear = []
results_temporal_spear = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom10/daily_precip/daysum_ICON_ngc3028_hourly_precip_30-60N_{year}.nc")    
    dry_da = ds["pr"]
    temporal_temp_pear = temporal_auto(dry_da,model)
    temporal_temp_pear = temporal_temp_pear.load()
    temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [year]})

    results_temporal_pear.append(temporal_temp_pear)

icon_cycle3_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")
icon_cycle3_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/icon_cycle3_30-60N_temporal_corr_nonbinary.nc')



## ICON cycle4

In [ ]:
model = "ICON"
years = range(2020, 2050)
months = [5,6,7,8,9,10]

results_temporal_pear = []
results_temporal_spear = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/NextGEMS/ICON_zoom9/2020-2050/daily_precip/30-60N_ICON_ngc4008_daily_precip_{year}.nc")
    ds = ds.where(ds.time.dt.month.isin(months), drop=True)
    dry_da = ds["pr"]
    temporal_temp_pear = temporal_auto(dry_da,model)
    temporal_temp_pear = temporal_temp_pear.load()
    temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [year]})

    results_temporal_pear.append(temporal_temp_pear)

icon_cycle4_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")
icon_cycle4_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/icon_cycle4_30-60N_temporal_corr_nonbinary.nc')


## IMERG

In [ ]:
model = "IMERG"

years = range(2014, 2024)

results_temporal_pear = []

for year in years:
    print(f"Processing year {year}...")
    ds = xr.open_dataset(f"/home/jwille/xenon_data/NextGEMS/IMERG/clean/global/QC/daysum/30-60N_daysum_QC_IMERG_V07B_final_{year}.nc4")  
    dry_da = ds["precipitation"]
    print(dry_da)
    temporal_temp_pear = temporal_auto(dry_da,model)
    temporal_temp_pear = temporal_temp_pear.load()
    temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [year]})

    results_temporal_pear.append(temporal_temp_pear)

imerg_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")
imerg_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/imerg_30-60N_temporal_corr_nonbinary.nc')



In [ ]:
print(dataset)

## CESM2

In [ ]:
# Change this to your directory path
model = 'CESM2'
#Source of CESM2 data /net/atmos/data/CESM2-LE/day/PRECT
#directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_past'
directory = '/home/jwille/xenon_data/NextGEMS/CESM2/precip_future'

# List all file names in the directory (adjust the filter as needed, e.g., for .nc files)
files = [f for f in os.listdir(directory) if f.endswith('.nc')]

# Create a dictionary to group files by simulation number
sim_files = defaultdict(list)


pattern = re.compile(r'LE2-1251\.(\d+)\.cam')

for file in files:
    match = pattern.search(file)
    if match:
        sim_num = match.group(1)
        sim_files[sim_num].append(file)
    else:
        print("Filename did not match expected pattern:", file)
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    # Here you can order or sort the files if the year range matters,
    # e.g., sorting by the start date in the filename (assuming YYYYMMDD is at a fixed position).
    # For example:
    file_list = sorted(file_list)

In [ ]:
# Now loop over each simulation number group and process the files for that simulation.
for sim, file_list in sim_files.items():
    print(f"Processing simulation {sim} with {len(file_list)} files:")
    latitude = [30,60]
    months = [5,6,7,8,9,10]
    results_temporal_pear = []
    results_temporal_spear = []
    
    file_list = sorted(file_list)
    # Create a full file path list
    full_paths = [os.path.join(directory, f) for f in file_list]

    #print(full_paths)
    dataset = xr.open_mfdataset(full_paths, engine='netcdf4',combine='by_coords')['PRECT']
    dataset = dataset * 86400 * 1000
    #dataset = dataset.sel(time=slice('1850', '2015'))  # select time range 
    dataset = dataset.sel(time=slice('2014', '2050'))  # select time range 
    dataset = dataset.where(dataset.time.dt.month.isin(months), drop=True)

    indexer = ((dataset.lat>=latitude[0]) & (dataset.lat<=latitude[1]))
    dataset = dataset.where(indexer,drop=True)
    dataset = dataset.convert_calendar('noleap')
    dataset = dataset.load()

    start_date = pd.Timestamp(''+str(dataset.time.dt.year.values[0]-1)+'-05-01')
    end_date = pd.Timestamp(''+str(dataset.time.dt.year.values[-1])+'-10-31')      
    year_range = dataset.time.dt.year.values[-1] - dataset.time.dt.year.values[0] + 1

    periods = pd.date_range(start_date, end_date, freq='Y')
    num_periods = len(periods)

    time_size = []
#    for i in range(num_periods - 1):
#        # Define the start and end dates for the current period
#        period_start = periods[i]
#        period_end = periods[i + 1]
#        dataset_sel = dataset.sel(time=slice(str(period_start),str(period_end)))
#        if len(dataset_sel.time) > 1:
#            time_size.append(len(dataset_sel.time))
#    print(dataset_sel.time)
#    result = np.zeros([len(time_size),len(dataset.lat),len(dataset.lon)])   


    
    for i in range(num_periods - 1):
        period_start = periods[i]
        period_end = periods[i + 1]
        print(period_start)
        print(period_end)
        dataset_sel = dataset.sel(time=slice(str(period_start),str(period_end)))
        if len(dataset_sel.time) > 1:
            temporal_temp_pear = temporal_auto(dataset_sel,model)

            temporal_temp_pear = temporal_temp_pear.expand_dims(dim={"year": [dataset_sel.time.values[0].year]})

            results_temporal_pear.append(temporal_temp_pear)

    cesm2_temporal_corr_pear = xr.concat(results_temporal_pear, dim="year")
    #cesm2_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_1850-2014_'+sim+'_30-60N_temporal_corr_nonbinary.nc')


    cesm2_temporal_corr_pear.to_netcdf(path='/home/jwille/xenon_data/NextGEMS/mid_lat_correlation/cesm2_2014-2049_'+sim+'_30-60N_temporal_corr_nonbinary.nc')
